# Starter Kit FahMai RAG

This notebook walks through building a minimalist **Retrieval-Augmented Generation (RAG)** system for the FahMai electronics store Q&A challenge.

**Sections:**
- **Section 0:** Setup & LLM Test
- **Section 1:** Dense Retrieval (MiniLM)
- **Section 2:** Sparse Retrieval (BM25)
- **Section 3:** Hybrid Retrieval (RRF)
- **Section 4:** What's Next?

In [ ]:
# Load the data from kaggle and upload here.
# !unzip data.zip

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# === CONFIGURATION ===
# Change N_QUESTIONS to 100 for a full competition run.

N_QUESTIONS = 100  # 10 for demo, 100 for real submission
DATA_DIR = "/content/drive/MyDrive/1.Training /2026/1.SuperAI/Hackathon/H3/data"
KB_DIR = f"{DATA_DIR}/knowledge_base"

---
## Section 0: Setup & LLM Test

First, install dependencies and test the ThaiLLM API — **without** any retrieval. This shows why RAG is needed.

ThaiLLM website: https://playground.thaillm.or.th/chat/

In [ ]:
!pip install -q sentence-transformers pythainlp rank-bm25 requests python-dotenv

In [ ]:
import os, csv, re, time, requests
import numpy as np
from pathlib import Path
from google.colab import userdata

THAILLM_API_KEY = userdata.get('ThaiLLM')

In [ ]:
def ask_llm(messages, model="typhoon", max_retries=5):
    """Call ThaiLLM API with retry and rate-limit handling.

    Available models: typhoon, openthaigpt, pathumma, kbtg
    """
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)

            # Rate limit — wait and retry
            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()

        except requests.exceptions.RequestException as e:
            wait = 2 ** attempt
            print(f"  Error: {e}, retrying in {wait}s...")
            time.sleep(wait)

    return None


def parse_answer(text):
    """Extract answer number from LLM response."""
    if text is None:
        return None
    # Remove any <think>...</think> blocks (some models do chain-of-thought)
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Look for ANSWER: X pattern
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m:
        return int(m.group(1))
    # Fallback: first standalone number 1-10
    for d in re.findall(r"\b(\d{1,2})\b", clean):
        if 1 <= int(d) <= 10:
            return int(d)
    return None

### Test the LLM without RAG

Let's ask a FahMai-specific question *without* any context. The LLM shouldn't know the answer.

In [ ]:
# Ask without context — LLM has no idea about FahMai's products
response = ask_llm([
    {"role": "user", "content": "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"}
])
print("LLM response (no context):", response)
print("\n→ The LLM doesn't know FahMai-specific facts. We need RAG!")

LLM response (no context): Watch S3 Ultra ของ Samsung ที่เปิดตัวในปี 2023 นั้น **กันน้ำได้ระดับ 5 ATM** หรือเทียบเท่ากับ **50 เมตร** ตามมาตรฐาน IPX8 ที่ระบุไว้ในข้อมูลผลิตภัณฑ์

ซึ่งหมายความว่า:

- สามารถกันน้ำได้ในระดับความลึกสูงสุด 50 เมตร
- สามารถใช้งานได้ในสภาพแวดล้อมที่มีน้ำ เช่น ว่ายน้ำ หรือล้างมือ
- ไม่ควรใช้ในกิจกรรมที่มีแรงดันน้ำสูง เช่น ดำน้ำลึก หรือใช้ในน้ำแรงดันสูง

อย่างไรก็ตาม ควรตรวจสอบข้อมูลจากเว็บไซต์ทางการของ Samsung หรือคู่มือการใช้งานเพื่อความแน่ใจ เพราะอาจมีการอัปเดตข้อมูลในอนาคต

สรุป: **Watch S3 Ultra กันน้ำได้ 5 ATM (50 เมตร)** ครับ ✅

→ The LLM doesn't know FahMai-specific facts. We need RAG!


### Load Questions

In [ ]:
questions = []
with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print(f"Loaded {len(questions)} questions, using first {N_QUESTIONS} for this run")
print(f"\nExample — Q1: {questions[0]['question']}")
for k, v in questions[0]["choices"].items():
    print(f"  {k}. {v}")

Loaded 100 questions, using first 100 for this run

Example — Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
  1. 3 ATM
  2. IP68
  3. 5 ATM
  4. IP67
  5. 10 ATM
  6. 20 ATM
  7. IPX8
  8. 1 ATM
  9. ไม่มีข้อมูลนี้ในฐานข้อมูล
  10. คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า


---
## Section 1: Dense Retrieval (MiniLM)

**Dense retrieval** converts text into vectors (embeddings) and finds relevant chunks by cosine similarity.

The pipeline: **Load docs → Chunk → Embed → Retrieve → Generate**

### 1.1 Load Knowledge Base

In [ ]:
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    documents.append({"path": str(fp.relative_to(kb_dir)), "text": text})

print(f"Loaded {len(documents)} documents")
print(f"  products/: {sum(1 for d in documents if 'products/' in d['path'])}")
print(f"  policies/: {sum(1 for d in documents if 'policies/' in d['path'])}")
print(f"  store_info/: {sum(1 for d in documents if 'store_info/' in d['path'])}")

# Preview one document
print(f"\n--- Sample: {documents[0]['path']} ---")
print(documents[0]["text"][:500])

Loaded 118 documents
  products/: 110
  policies/: 5
  store_info/: 3

--- Sample: policies/cancellation_policy.md ---
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ้นอยู่กับสถานะคำสั่งซื้อเป็นหลัก

---

## 2. การยกเลิกตามสถานะคำสั่งซื้อ

### 2.1 สถานะ "รอชำระเงิน" (Pending Payment)

**ยกเลิกได้ทันที**

คำสั่งซื้อที่อยู่ในสถานะรอชำระเงินสามารถยกเลิกได้ทันทีโดยไม่มีค่าใช้จ่าย ผ่านแอปพลิ


### 1.2 Chunking

LLMs have limited context windows, and long documents dilute relevance. We split each document into smaller **chunks** using a fixed-size sliding window with overlap.

In [ ]:
# 1. ขยายขนาด Chunk ให้คลุมประโยคภาษาไทยได้จบความมากขึ้น
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

def make_chunks(text, size, overlap):
    """Split text into overlapping fixed-size windows."""
    if len(text) <= size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start : start + size])
        start += size - overlap
    return chunks

# Build all chunks
chunks = []
for doc in documents:
    for window in make_chunks(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP):
        # 🔴 การอัปเกรดที่สำคัญที่สุด: ฝังชื่อไฟล์ (Path) นำหน้าเนื้อหาเสมอ!
        # เช่น "[ข้อมูลจาก: products/SaiFah_X1.md]\nความจุแบตเตอรี่..."
        enriched_text = f"[ข้อมูลจาก: {doc['path']}]\n{window}"

        chunks.append({"text": enriched_text, "source": doc["path"]})

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\n--- Sample chunk ---")
print(f"Source: {chunks[0]['source']}")
print(chunks[0]["text"][:300]) # ลองปริ้นท์ดู จะเห็นว่ามี [ข้อมูลจาก: ...] นำหน้าแล้ว

Created 650 chunks from 118 documents

--- Sample chunk ---
Source: policies/cancellation_policy.md
[ข้อมูลจาก: policies/cancellation_policy.md]
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ค


### 1.3 Embedding

We use `paraphrase-multilingual-MiniLM-L12-v2` — a small (118M params), multilingual embedding model that produces 384-dimensional vectors. It supports Thai out of the box and doesn't require any special prefixes.

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.4 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1. เปลี่ยน Model เป็นรุ่นที่เข้าใจความหมายและบริบทภาษาไทยได้ลึกซึ้งขึ้น
print("Loading Embedding Model: paraphrase-multilingual-mpnet-base-v2...")
embedding_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# 2. เตรียมข้อมูล: ดึงเฉพาะส่วน "text" ออกมาจาก chunks ที่เราฝังชื่อไฟล์ไว้ในขั้น 1.2
texts_to_encode = [chunk["text"] for chunk in chunks]

# 3. แปลงข้อความเป็น Vector
print(f"Encoding {len(texts_to_encode)} chunks into vectors...")
dense_embeddings = embedding_model.encode(texts_to_encode, show_progress_bar=True)

# 4. สร้างฐานข้อมูล Vector (FAISS Index)
# ให้โค้ดอ่านขนาด Dimension อัตโนมัติ (mpnet จะได้ 768)
dimension = dense_embeddings.shape[1]
print(f"Vector Dimension: {dimension}")

# สร้างกล่อง FAISS และยัด Vector ทั้งหมดลงไป
dense_index = faiss.IndexFlatL2(dimension)
dense_index.add(np.array(dense_embeddings).astype('float32'))

print(f"✅ FAISS Index สร้างเสร็จสมบูรณ์!")
print(f"✅ มีเอกสารพร้อมค้นหาทั้งหมด: {dense_index.ntotal} chunks")

Loading Embedding Model: paraphrase-multilingual-mpnet-base-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 650 chunks into vectors...


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Vector Dimension: 768
✅ FAISS Index สร้างเสร็จสมบูรณ์!
✅ มีเอกสารพร้อมค้นหาทั้งหมด: 650 chunks


### 1.4 Retrieve

Embed the question, then find the most similar chunks via dot product (= cosine similarity for normalized vectors).

In [ ]:
import numpy as np

# 1. เพิ่มจำนวน Top-K: จาก 5 เป็น 7 เพื่อให้ Typhoon มีข้อมูลประกอบการตัดสินใจมากขึ้น
TOP_K = 7

def dense_retrieve(query_text, chunk_vectors, k=TOP_K):
    """Return indices of top-k most similar chunks to the query."""
    q_emb = embedding_model.encode([query_text], normalize_embeddings=True)
    scores = np.dot(chunk_vectors, q_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# --- Demo: ทดสอบระบบดึงข้อมูลด้วยเทคนิคลับ ---
q = questions[0]

# ดึงตัวเลือกจาก q['choices']
choices_text = " ".join([str(q['choices'][str(i)]) for i in range(1, 9)])
enhanced_query = f"{q['question']} {choices_text}"

idx, scores = dense_retrieve(enhanced_query, dense_embeddings)

print(f"คำถามดั้งเดิม: {q['question']}")
print(f"คำถามที่ถูกอัปเกรด (Enhanced Query): {enhanced_query[:150]}...\n")
print("--- ผลการค้นหา ---")
for rank, (i, s) in enumerate(zip(idx, scores), 1):
    print(f"Rank {rank} (ความแม่นยำ: {s:.3f}) {chunks[i]['source']}")
    print(f"{chunks[i]['text'][:150]}...\n")

คำถามดั้งเดิม: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
คำถามที่ถูกอัปเกรด (Enhanced Query): Watch S3 Ultra กันน้ำได้กี่ ATM ครับ 3 ATM IP68 5 ATM IP67 10 ATM 20 ATM IPX8 1 ATM...

--- ผลการค้นหา ---
Rank 1 (ความแม่นยำ: 1.392) policies/warranty_policy.md
[ข้อมูลจาก: policies/warranty_policy.md]
-----|
| ตัวเครื่อง | **1 ปี** |
| สายนาฬิกา / Charger | **6 เดือน** |

ครอบคลุม: Watch S3 Series (S3, S3 Pro...

Rank 2 (ความแม่นยำ: 1.241) products/WK-SW-002_wongkhojon_watch_s3_pro.md
[ข้อมูลจาก: products/WK-SW-002_wongkhojon_watch_s3_pro.md]
มีนาคม 2569)

Points ที่ได้รับจะถูกเครดิตเข้าบัญชีภายใน 7 วันทำการหลังจากได้รับสินค้า

---
...

Rank 3 (ความแม่นยำ: 1.232) products/WK-SW-008_wongkhojon_watch_s2_pro.md
[ข้อมูลจาก: products/WK-SW-008_wongkhojon_watch_s2_pro.md]
น้าจอมีความสว่างสูงสุด 1,000 nits แทน 1,600 nits ของ S3 Pro

**ความแตกต่างสำคัญที่ต้องทราบก...

Rank 4 (ความแม่นยำ: 1.215) products/SF-SP-010_saifah_phone_x9_fe.md
[ข้อมูลจาก: products/SF-SP-010_saifah_phone_x9_fe.md]
|
| ชิปเซ็ต | Sa

### 1.5 Generate Answer

Send the retrieved context + question + choices to the LLM and parse the answer.

In [ ]:
# === อัปเกรด SYSTEM_PROMPT ให้ฉลาดและดักทางตัวเลือก 9, 10 ===
SYSTEM_PROMPT = """คุณคือ AI ผู้ช่วยบริการลูกค้าของ "ร้านฟ้าใหม่" (FahMai) ร้านขายเครื่องใช้ไฟฟ้า
หน้าที่ของคุณคือตอบคำถามลูกค้า โดยเลือกตัวเลือก (1-10) ที่ถูกต้องที่สุดเพียงข้อเดียว

กฎเหล็ก:
1. ตอบด้วยรูปแบบ ANSWER: X เท่านั้น (โดย X คือตัวเลข 1 ถึง 10) ห้ามมีคำอธิบายอื่นเจือปน
2. วิเคราะห์ข้อมูลจาก 'Context' ที่กำหนดให้เท่านั้น ห้ามคิดเอาเอง
3. หากใน 'Context' ไม่มีข้อมูลเพียงพอที่จะตอบคำถามได้อย่างมั่นใจ ให้ตอบตัวเลือกที่ระบุว่า "ไม่มีข้อมูลนี้ในฐานข้อมูล" (ช้อยส์ 9)
4. หากคำถามนั้นไม่เกี่ยวกับการซื้อขายสินค้า, บริการ, นโยบาย ของร้านฟ้าใหม่เลย ให้ตอบตัวเลือกที่ระบุว่า "คำถามนี้ไม่เกี่ยวข้อง" (ช้อยส์ 10)
5. ระวังชื่อแบรนด์ House Brands: สายฟ้า (SaiFah), ดาวเหนือ (DaoNuea), คลื่นเสียง (KluenSiang), วงโคจร (WongKhoJon), จุดเชื่อม (JudChuem)"""

def build_rag_prompt(question, choices, retrieved_chunks):
    """Build the user prompt with retrieved context."""
    # จัดเรียง Context ให้มีเส้นคั่นอ่านง่ายขึ้น
    context = "\n\n".join(
        f"--- ข้อมูลอ้างอิงที่ {i+1} ---\n{c['text']}"
        for i, c in enumerate(retrieved_chunks)
    )
    # จัดเรียงตัวเลือก
    choices_text = "\n".join(f"{k}. {v}" for k, v in choices.items())

    # === CUSTOMIZED PROMPT ===
    return (
        f"ข้อมูลสำหรับใช้อ้างอิง (Context):\n{context}\n\n"
        f"คำถาม: {question}\n\n"
        f"ตัวเลือก:\n{choices_text}\n\n"
        f"จากข้อมูลอ้างอิงด้านบน คำตอบที่ถูกต้องที่สุดคือตัวเลือกหมายเลขใด?\n"
        f"ตอบ ANSWER: X (X คือหมายเลขตัวเลือก 1-10)"
    )

# Demo: answer Q1
q = questions[0]

# รวบรวมช้อยส์ 1 ถึง 8 มาต่อท้ายคำถาม เพื่อช่วยในการค้นหา (Query Expansion)
choices_for_search = " ".join([str(q['choices'][str(i)]) for i in range(1, 9)])
enhanced_query = f"{q['question']} {choices_for_search}"

# ใช้คำถามที่อัปเกรดแล้วไปค้นหา
idx, _ = dense_retrieve(enhanced_query, dense_embeddings)
retrieved = [chunks[i] for i in idx]

# สร้าง Prompt และเรียก LLM
prompt = build_rag_prompt(q["question"], q["choices"], retrieved)

print("กำลังส่งคำถามให้ Typhoon ประมวลผล...\n")
raw = ask_llm([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": prompt},
])
answer = parse_answer(raw)

print(f"Q{q['id']}: {q['question']}")
print(f"LLM raw: {raw}")
print(f"Parsed answer: {answer}")

กำลังส่งคำถามให้ Typhoon ประมวลผล...

Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
LLM raw: ANSWER: 5
Parsed answer: 5


In [ ]:
retrieved

[{'text': '[ข้อมูลจาก: policies/warranty_policy.md]\n-----|\n| ตัวเครื่อง | **1 ปี** |\n| สายนาฬิกา / Charger | **6 เดือน** |\n\nครอบคลุม: Watch S3 Series (S3, S3 Pro, S3 SE, S3 Ultra), Watch S2 Series, Ring 1\n\n### 2.6 วงโคจร (WongKojorn) — ฟิตเนสแทร็กเกอร์\n\n| ชิ้นส่วน | ระยะเวลาประกัน |\n|---------|---------------|\n| ตัวเครื่อง | **1 ปี** |\n| สาย | **6 เดือน** |\n\nครอบคลุม: Band 8, Band 8 Pro, Band 7, Band 6\n\n### 2.7 จุดเชื่อม (JudChuem) — อุปกรณ์เสริมทั่วไป\n\n| ประเภทสินค้า | ระยะเวลาประกัน |\n|-------------|---------------|\n| อุปกรณ์เสริมทั่วไป (แท่นชาร์จ, ชาร์จเจอร์, Hub, Dock, เคส, ฟิล์ม) | **6 เดือน** |\n| สาย USB / สายชาร์จ | **3 เดือน** |\n\nครอบคลุม: ชาร์จเจอร์ GaN Series, QiPad 15, Hub USB-C 7-in-1, Dock AirBook Edition, สายเคเบิล USB-C ทุกขนาด, สาย USB-C to Lightning, เคสสมาร์ทโฟน, SaiFah Pen Gen 2\n\n---\n\n## 3. ระยะเวลาการรับประกัน — แบรนด์พา',
  'source': 'policies/warranty_policy.md'},
 {'text': '[ข้อมูลจาก: products/WK-SW-002_wongkhojon_watch_s3_pro.md]\nมีน

### 1.6 Run All Questions (Dense)

Loop through questions, retrieve, generate, and collect predictions.

In [ ]:
import time

def run_pipeline(questions, retrieve_fn, label="dense", n=N_QUESTIONS):
    """Run retrieval + LLM on first n questions. Returns predictions dict."""
    predictions = {}

    print(f"กำลังรันระบบ {label} จำนวน {n} ข้อ... (อาจจะใช้เวลาสักครู่)")

    for i, q in enumerate(questions[:n]):
        # 🔴 อัปเกรด 1: ใช้เทคนิค Query Expansion กับทุกข้อ!
        # เอาช้อยส์ 1-8 มาต่อท้ายเพื่อช่วย AI ค้นหาเอกสารได้แม่นยำขึ้น
        choices_for_search = " ".join([str(q['choices'][str(c)]) for c in range(1, 9)])
        enhanced_query = f"{q['question']} {choices_for_search}"

        # ส่ง enhanced_query ไปค้นหาแทนการใช้คำถามสั้นๆ
        retrieved_chunks = retrieve_fn(enhanced_query)

        # สร้าง Prompt (ส่งคำถามดั้งเดิมไปให้ LLM อ่าน จะได้ไม่งง)
        prompt = build_rag_prompt(q["question"], q["choices"], retrieved_chunks)

        raw = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ])

        pred = parse_answer(raw)

        # 🔴 อัปเกรด 2: ถ้าสกัดตัวเลขไม่ได้ ให้ตอบ 9 (ไม่มีข้อมูล) แทนการเดาช้อยส์ 1
        predictions[q["id"]] = pred if pred else 9

        print(f"  Q{q['id']:>3}: pred={predictions[q['id']]}")
        time.sleep(0.3)  # หน่วงเวลาเพื่อไม่ให้ API โดนแบน (Rate Limit)

    print(f"\n✅ {label}: ตอบคำถามเสร็จสิ้น {len(predictions)} ข้อ")
    return predictions

# Dense retrieval function
def dense_retrieve_chunks(query_text):
    # 🔴 อัปเกรด 3: ใช้ตัวแปร dense_embeddings ตามที่เราตั้งชื่อไว้ใน Section 1.3
    idx, _ = dense_retrieve(query_text, dense_embeddings)
    return [chunks[i] for i in idx]

# --- สั่งรันจริง! ---
dense_preds = run_pipeline(questions, dense_retrieve_chunks, label="dense")

กำลังรันระบบ dense จำนวน 100 ข้อ... (อาจจะใช้เวลาสักครู่)
  Q  1: pred=5
  Error: 520 Server Error: <none> for url: http://thaillm.or.th/api/typhoon/v1/chat/completions, retrying in 1s...
  Q  2: pred=7
  Q  3: pred=2
  Q  4: pred=9
  Q  5: pred=9
  Q  6: pred=8
  Q  7: pred=8
  Q  8: pred=4
  Q  9: pred=9
  Q 10: pred=2
  Q 11: pred=4
  Q 12: pred=1
  Q 13: pred=6
  Q 14: pred=3
  Q 15: pred=1
  Q 16: pred=1
  Q 17: pred=8
  Q 18: pred=5
  Q 19: pred=2
  Q 20: pred=2
  Q 21: pred=3
  Q 22: pred=6
  Q 23: pred=3
  Q 24: pred=8
  Error: 520 Server Error: <none> for url: http://thaillm.or.th/api/typhoon/v1/chat/completions, retrying in 1s...
  Q 25: pred=5
  Q 26: pred=6
  Q 27: pred=2
  Q 28: pred=7
  Q 29: pred=2
  Q 30: pred=3
  Q 31: pred=4
  Q 32: pred=2
  Q 33: pred=8
  Q 34: pred=5
  Q 35: pred=3
  Q 36: pred=2
  Q 37: pred=8
  Q 38: pred=6
  Q 39: pred=4
  Q 40: pred=8
  Q 41: pred=7
  Q 42: pred=9
  Q 43: pred=9
  Q 44: pred=1
  Q 45: pred=2
  Q 46: pred=1
  Q 47: pred=2
  Q 48:

---
## Section 2: Sparse Retrieval (BM25)

**BM25** is a keyword-matching algorithm. It scores documents by how many query terms they contain, weighted by term rarity (IDF). No neural network needed.

### 2.1 Thai Tokenization

BM25 needs tokenized text. Thai has no spaces between words, so we use `pythainlp` to segment.

In [ ]:
!pip install pythainlp rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 85.5 MB/s eta 0:00:00


In [ ]:
from pythainlp.tokenize import word_tokenize

# อัปเกรด: สร้างฟังก์ชันตัดคำเฉพาะสำหรับทำ BM25
def tokenize_for_bm25(text):
    if not isinstance(text, str):
        return []

    # 1. แปลงตัวอักษรภาษาอังกฤษให้เป็นพิมพ์เล็กทั้งหมด (ช่วยจับคู่คำได้ดีขึ้น)
    text = text.lower()

    # 2. ตัดคำด้วย newmm และใส่ keep_whitespace=False เพื่อโยนช่องว่างทิ้งไป
    tokens = word_tokenize(text, engine="newmm", keep_whitespace=False)

    return tokens

# Demo: ทดสอบเปรียบเทียบผลลัพธ์
sample = "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"

# ลองเรียกใช้ฟังก์ชันที่เราอัปเกรด
tokens = tokenize_for_bm25(sample)

print("--- ทดสอบระบบตัดคำ (Tokenization) ---")
print(f"ข้อความต้นฉบับ: {sample}")
print(f"Tokens ที่ได้: {tokens}")

--- ทดสอบระบบตัดคำ (Tokenization) ---
ข้อความต้นฉบับ: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?
Tokens ที่ได้: ['watch', 's', '3', 'ultra', 'กันน้ำ', 'ได้', 'กี่', 'atm', 'ครับ', '?']


### 2.2 Build BM25 Index

In [ ]:
from rank_bm25 import BM25Okapi

# 1. ตัดคำทุก Chunk โดยใช้ฟังก์ชันที่อัปเกรดแล้ว (จัดการตัวพิมพ์เล็กและลบช่องว่าง)
print("กำลังตัดคำสำหรับสร้าง BM25 Index...")
tokenized_chunks = [tokenize_for_bm25(c["text"]) for c in chunks]

# 2. สร้าง BM25 Index
bm25 = BM25Okapi(tokenized_chunks)

print(f"✅ สร้าง BM25 index สำเร็จ ครอบคลุมทั้งหมด {len(tokenized_chunks)} chunks")

กำลังตัดคำสำหรับสร้าง BM25 Index...
✅ สร้าง BM25 index สำเร็จ ครอบคลุมทั้งหมด 650 chunks


### 2.3 Retrieve with BM25

Compare BM25 results with dense results for the same question.

In [ ]:
def bm25_retrieve(query_text, k=TOP_K):
    """Return top-k chunk indices using BM25 with upgraded tokenization."""
    # 🔴 อัปเกรด: ใช้ฟังก์ชันตัดคำที่ทำ lower() และตัดช่องว่างทิ้งที่เราสร้างไว้
    tokens = tokenize_for_bm25(query_text)

    # คำนวณคะแนนด้วย BM25 Index ที่เราสร้างไว้ใน 2.2
    scores = bm25.get_scores(tokens)

    # เรียงลำดับคะแนนจากมากไปน้อย
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# --- ส่วนของการเปรียบเทียบ (Comparison) ---
q = questions[0]

# 🔴 เทคนิคพิเศษ: ใช้คำถามที่รวมตัวเลือก (Enhanced Query) เพื่อการเปรียบเทียบที่ยุติธรรม
choices_for_search = " ".join([str(q['choices'][str(i)]) for i in range(1, 9)])
enhanced_query = f"{q['question']} {choices_for_search}"

print(f"คำถามที่ใช้ค้นหา: {q['question']}\n")

# 1. ค้นหาด้วย Dense (ความหมาย) - ใช้ตัวแปรที่เราตั้งชื่อใหม่ใน 1.3/1.4
d_idx, _ = dense_retrieve(enhanced_query, dense_embeddings)

# 2. ค้นหาด้วย BM25 (คีย์เวิร์ด)
b_idx, _ = bm25_retrieve(enhanced_query)

print(f"--- Dense Retrieval Top-{TOP_K} (เน้นความหมาย) ---")
for i in d_idx:
    print(f"  [{chunks[i]['source']}]")

print(f"\n--- BM25 Retrieval Top-{TOP_K} (เน้นคีย์เวิร์ดตรงตัว) ---")
for i in b_idx:
    print(f"  [{chunks[i]['source']}]")

คำถามที่ใช้ค้นหา: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

--- Dense Retrieval Top-7 (เน้นความหมาย) ---
  [policies/warranty_policy.md]
  [products/WK-SW-002_wongkhojon_watch_s3_pro.md]
  [products/WK-SW-008_wongkhojon_watch_s2_pro.md]
  [products/SF-SP-010_saifah_phone_x9_fe.md]
  [products/SF-SP-003_saifah_phone_x9.md]
  [products/SF-SP-001_saifah_phone_x9_pro_max.md]
  [policies/warranty_policy.md]

--- BM25 Retrieval Top-7 (เน้นคีย์เวิร์ดตรงตัว) ---
  [products/WK-SW-002_wongkhojon_watch_s3_pro.md]
  [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
  [products/WK-SW-003_wongkhojon_watch_s3.md]
  [products/WK-SW-008_wongkhojon_watch_s2_pro.md]
  [products/WK-SW-006_wongkhojon_watch_classic_c1.md]
  [products/WK-SW-002_wongkhojon_watch_s3_pro.md]
  [products/WK-SW-004_wongkhojon_watch_s3_se.md]


### 2.4 Run All Questions (BM25)

In [ ]:
# 1. สร้าง Wrapper Function สำหรับดึงข้อมูลแบบ BM25
def bm25_retrieve_chunks(query_text):
    # ใช้ฟังก์ชัน bm25_retrieve ที่เราอัปเกรดระบบตัดคำไว้ใน 2.3
    idx, _ = bm25_retrieve(query_text)
    return [chunks[i] for i in idx]

# 2. สั่งรัน Pipeline สำหรับ BM25 ครบทุกข้อ (N_QUESTIONS)
# ระบบจะใช้ Enhanced Query อัตโนมัติจากฟังก์ชัน run_pipeline ที่เราแก้ไว้ก่อนหน้า
print("--- เริ่มต้นการรัน Pipeline ด้วยระบบ BM25 (Keyword Search) ---")
bm25_preds = run_pipeline(questions, bm25_retrieve_chunks, label="bm25")

# 3. สรุปผล
print(f"✅ BM25: เก็บคำทำนายครบ {len(bm25_preds)} ข้อเรียบร้อยแล้ว")

--- เริ่มต้นการรัน Pipeline ด้วยระบบ BM25 (Keyword Search) ---
กำลังรันระบบ bm25 จำนวน 100 ข้อ... (อาจจะใช้เวลาสักครู่)
  Q  1: pred=5
  Q  2: pred=7
  Q  3: pred=2
  Q  4: pred=6
  Q  5: pred=6
  Q  6: pred=8
  Q  7: pred=1
  Q  8: pred=9
  Q  9: pred=4
  Q 10: pred=2
  Q 11: pred=4
  Q 12: pred=1
  Q 13: pred=2
  Q 14: pred=3
  Q 15: pred=7
  Q 16: pred=1
  Q 17: pred=8
  Q 18: pred=5
  Q 19: pred=2
  Q 20: pred=2
  Q 21: pred=3
  Q 22: pred=6
  Q 23: pred=3
  Q 24: pred=3
  Q 25: pred=5
  Q 26: pred=6
  Q 27: pred=2
  Q 28: pred=7
  Q 29: pred=4
  Q 30: pred=3
  Q 31: pred=4
  Q 32: pred=2
  Q 33: pred=8
  Q 34: pred=5
  Q 35: pred=3
  Q 36: pred=2
  Q 37: pred=8
  Q 38: pred=6
  Q 39: pred=4
  Q 40: pred=8
  Q 41: pred=7
  Q 42: pred=2
  Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/typhoon/v1/chat/completions, retrying in 1s...
  Q 43: pred=4
  Q 44: pred=1
  Q 45: pred=3
  Q 46: pred=1
  Error: 520 Server Error: <none> for url: http://thaillm.or.th/api/ty

---
## Section 3: Hybrid Retrieval (RRF)

**Hybrid** combines dense and sparse results. The idea: dense is good at semantic matching, BM25 is good at exact keyword matching. Together they cover more cases.

We use **Reciprocal Rank Fusion (RRF)** to merge the two ranked lists:

$$\text{score}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k$ is a constant (typically 60). Documents ranked highly by *either* method get a high combined score.

In [ ]:
def hybrid_retrieve(query_text, chunk_vectors, k=7, rrf_k=60):
    fetch_k = 20
    d_idx, _ = dense_retrieve(query_text, chunk_vectors, k=fetch_k)
    b_idx, _ = bm25_retrieve(query_text, k=fetch_k)

    rrf_scores = {}
    # ให้ค่าน้ำหนัก Dense (ความหมาย) มากกว่า BM25 เล็กน้อย (1.2 vs 1.0)
    for rank, idx in enumerate(d_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.2 / (rrf_k + rank)
    for rank, idx in enumerate(b_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)

    sorted_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:k]
    return sorted_idx

### 3.2 Run All Questions (Hybrid)

In [ ]:
# 1. สร้าง Wrapper Function สำหรับดึงข้อมูลแบบ Hybrid (RRF)
def hybrid_retrieve_chunks(query_text):
    #ใช้ตัวแปร dense_embeddings และฟังก์ชัน hybrid_retrieve ที่จูนไว้ใน 3.1
    idx = hybrid_retrieve(query_text, dense_embeddings)
    return [chunks[i] for i in idx]

# 2. สั่งรัน Pipeline สำหรับ Hybrid ครบ 100 ข้อ (N_QUESTIONS)
print("--- [FINAL RUN] เริ่มต้นการรัน Pipeline ด้วยระบบ Hybrid (Dense + BM25) ---")
hybrid_preds = run_pipeline(questions, hybrid_retrieve_chunks, label="hybrid")

# 3. สรุปผลการทำงาน
print(f"\n✅ Hybrid: ทำนายคำตอบครบ {len(hybrid_preds)} ข้อเรียบร้อยแล้ว!")
print("ตอนนี้คุณมีคำทำนายที่แม่นยำที่สุดอยู่ในตัวแปร hybrid_preds แล้วครับ")

--- [FINAL RUN] เริ่มต้นการรัน Pipeline ด้วยระบบ Hybrid (Dense + BM25) ---
กำลังรันระบบ hybrid จำนวน 100 ข้อ... (อาจจะใช้เวลาสักครู่)
  Error: 522 Server Error: <none> for url: http://thaillm.or.th/api/typhoon/v1/chat/completions, retrying in 1s...
  Q  1: pred=5
  Q  2: pred=7
  Q  3: pred=2
  Q  4: pred=6
  Q  5: pred=6
  Q  6: pred=8
  Q  7: pred=1
  Q  8: pred=4
  Q  9: pred=4
  Q 10: pred=2
  Q 11: pred=4
  Q 12: pred=1
  Q 13: pred=6
  Q 14: pred=3
  Q 15: pred=1
  Q 16: pred=1
  Q 17: pred=8
  Q 18: pred=5
  Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/typhoon/v1/chat/completions, retrying in 1s...
  Q 19: pred=2
  Q 20: pred=2
  Q 21: pred=3
  Error: 520 Server Error: <none> for url: http://thaillm.or.th/api/typhoon/v1/chat/completions, retrying in 1s...
  Q 22: pred=6
  Q 23: pred=3
  Q 24: pred=3
  Q 25: pred=5
  Q 26: pred=6
  Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/typhoon/v1/chat/completions, retrying in 1s...
  Q 27

### 3.3 Compare All Three Methods

In [ ]:
print(f"{'QID':>4}  {'Dense':>6} {'BM25':>6} {'Hybrid':>7}")
print("-" * 30)
for q in questions[:N_QUESTIONS]:
    qid = q["id"]
    d = dense_preds.get(qid, "-")
    b = bm25_preds.get(qid, "-")
    h = hybrid_preds.get(qid, "-")
    match = "" if d == b == h else "  ← differ"
    print(f"Q{qid:>3}  {d:>6} {b:>6} {h:>7}{match}")

 QID   Dense   BM25  Hybrid
------------------------------
Q  1       5      5       5
Q  2       7      7       7
Q  3       2      2       2
Q  4       9      6       6  ← differ
Q  5       9      6       6  ← differ
Q  6       8      8       8
Q  7       8      1       1  ← differ
Q  8       4      9       4  ← differ
Q  9       9      4       4  ← differ
Q 10       2      2       2
Q 11       4      4       4
Q 12       1      1       1
Q 13       6      2       6  ← differ
Q 14       3      3       3
Q 15       1      7       1  ← differ
Q 16       1      1       1
Q 17       8      8       8
Q 18       5      5       5
Q 19       2      2       2
Q 20       2      2       2
Q 21       3      3       3
Q 22       6      6       6
Q 23       3      3       3
Q 24       8      3       3  ← differ
Q 25       5      5       5
Q 26       6      6       6
Q 27       2      2       2
Q 28       7      7       7
Q 29       2      4       4  ← differ
Q 30       3      3       3
Q 31       

### Write Submission

Pick your best method and generate a `submission.csv` for Kaggle.

> Set `N_QUESTIONS = 100` at the top and re-run the notebook to generate a full submission.

In [ ]:
# Use dense predictions as the submission (change to hybrid_preds or bm25_preds to try others)
best_preds = hybrid_preds

with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions:
        qid = q["id"]
        writer.writerow([qid, best_preds.get(qid, 1)])  # default=1 for unanswered

print(f"submission.csv written ({len(questions)} rows)")

submission.csv written (100 rows)


---
## Section 4: What's Next?

This baseline uses a simple setup. Here are ideas to improve your score:

- **Better embeddings** — try other, stronger multilingual  embedding.
- **Smarter chunking** — split by structure or other methods or add useful information to each chunk
- **Chunk size tuning** — experiment with  256, 512, 1024 or something else character chunks
- **Different ThaiLLMs** — try `openthaigpt`, `kbtg`, `pathumma`.
- **Prompt engineering** — adjust the system prompt, add few-shot examples, or change the output format
- **Reranking** — use a cross-encoder or specialized reranker to re-score the top-k chunks before sending to the

**Feel free to implement your own RAG.**